# Diffie-Hellman Key Exchange

Diffie-Hellman is a Key Exchange Algorithm based on the principles of **Modular Exponentiation** and **Discrete Logarithm Problem**. It enables two entities to agree upon a secret key without prior arrangements, even in the presence of a potential eavesdroppers. The key is then used for encryption and decryption, which is vital for maintaining confidentiality in communication.

$\boxed{Note}$  A key exchange algorithm helps in preserving the integrity of the transmitted data, it prevents unauthorized alteration or tampering of data during transmission.

## Primitive Root

$\alpha$ is said to be a *primitive root* of prime number $p$, if $\alpha^{1}$ mod $p$, $\alpha^{2}$ mod $p$, $\alpha^{3}$ mod $p$, ... , $\alpha^{p-1}$ mod $p$ are distinct.

Examples:

**Is 2 a primitive root of 5?**

|   |   |   |
|---|---|---|
|$2^1$ mod $5$|$2$ mod $5$|$\color{green}{2}$|
|$2^2$ mod $5$|$4$ mod $5$|$\color{green}{4}$|
|$2^3$ mod $5$|$8$ mod $5$|$\color{green}{8}$|
|$2^4$ mod $5$|$16$ mod $5$|$\color{green}{6}$|

Since all possible values are distinct, 2 is indeed a primitive root of 5.

**Is 3 a primitive root of 7?**

|   |   |   |
|---|---|---|
|$3^1$ mod $7$|$3$ mod $7$|$\color{green}{3}$|
|$3^2$ mod $7$|$6$ mod $7$|$\color{green}{2}$|
|$3^3$ mod $7$|$9$ mod $7$|$\color{green}{6}$|
|$3^4$ mod $7$|$27$ mod $7$|$\color{green}{4}$|
|$3^5$ mod $7$|$243$ mod $7$|$\color{green}{5}$|
|$3^6$ mod $7$|$729$ mod $7$|$\color{green}{1}$|

Since all possible values are distinct, 3 is indeed a primitive root of 7.

**Is 2 a primitive root of 7?**

|   |   |   |
|---|---|---|
|$2^1$ mod $7$|$2$ mod $7$|$\color{green}{2}$|
|$2^2$ mod $7$|$4$ mod $7$|$\color{green}{4}$|
|$2^3$ mod $7$|$8$ mod $7$|$\color{green}{1}$|
|$2^4$ mod $7$|$16$ mod $7$|$\color{red}{2}$|
|$2^5$ mod $7$|$32$ mod $7$|$\color{red}{4}$|
|$2^6$ mod $7$|$64$ mod $7$|$\color{red}{1}$|

Since all possible values are distinct, 2 is NOT a primitive root of 7.

## Algorithm

1. Party A and B must decide upon two numbers on a public channel. These two number are shared and are not kept secret.
    - A large prime number $p$
    - A generator $g$ of $p$, which is primitive root of $p$
2. A and B must now choose their own respective private keys
    - Let $a$ be the private key of A
    - Let $b$ be the private key of B
3. They must now produce a public key and then share it with each other (over a public channel)
    - $P_a$: $g^a$ mod $p$
    - $P_b$: $g^b$ mod $p$
4. They can now have the same secret key by using modular exponentiation:
    - Private key of A: ($g^a$ mod $p$)$^b$ = $g^{ab}$ mod $p$
    - Private key of B: ($g^b$ mod $p$)$^a$ = $g^{ab}$ mod $p$

$$\boxed{Private\texttt{ }Key = g^{ab}\texttt{ }mod\texttt{ }p}$$

![Algorithm](Diffie_Hellman.png)

## Strength of Diffie-Hellman Key Exchange Algorithm

The Diffie-Hellman key exchange is secure because of the difficulty of calculation discrete logarithms. An eavesdropper listening to the communication channel for the exchanged value of $P_a$ and $P_b$ would find it extremely difficult to determine shared secret without knowing the value of $a$ or $b$ which are private keys and a limited to the one party.

## Perfect Forward Secrecy (PFS)

Perfect Forward Secrecy is the property in the cryptography that prevents the exposure of long-term secret keys from compromising the past or future communication. In context to Diffie-Hellman, prefect forward secrecy means that even if an attacker were somehow gain/compute the private keys used during a session, he would not be able to decrypt past communications or use those keys to decrypt any of the future communication.

### Applications:

- **Use of Session Keys**: Systems that implements PFS generates a unique session key for every session, so even if an attackers manages to know the current session key, it cannot use it to decrypt past or future communications, as each session keys becomes invalid after session is over.
- **Temporary Keys**: The use of temporary keys generated by PFS system for each session, which is not use for other sessions, ensuring that if one key is compromised, it doesn't compromises other communications.
- **Zero Dependence on Long-term Keys**: The long-term keys used for authentication or key exchange, in case they are compromised other communication remains secured. As they are used for the purpose of establishing the session keys.
- **Enhanced Security**: PFS add a layer to the security, in scenarios where long-term keys might be at risk due to various factors such as complex cyber attacks, compromised servers or future cryptographic development.

## Implementation

In [13]:
import sympy
import os 

def generate():
    '''
    This function generates a large prime number p and a primitive root g of p.
    '''
    p = sympy.randprime(10**8, 10**16) # Generate a random prime number between 10^8 and 10^16
    g = sympy.primitive_root(p) # Find a primitive root modulo p
    return p, g

prime, generator = generate() # Generate a large prime number p and a primitive root g of p
if not os.path.exists('.env'):
    open('.env', 'x').close() # Create an empty .env file if it doesn't exist

with open('.env', 'w') as f:
    f.write(f'p={prime}\n')
    f.write(f'g={generator}\n')

print(f'Public value of prime p: {prime}')
print(f'Public value of prime g: {generator}')

Public value of prime p: 2149052623503911
Public value of prime g: 11


In [14]:
# Let two parties be A and B
class DiffieHellman:
    p = g = 0
    private_key = 0
    shared_key = 0
    def __init__(self):
        self.private_key = sympy.randprime(10**8, 10**16) # Generate a random private key for A

    def get_public_key(self):
        with open('.env', 'r') as f:
            for line in f:
                if line.startswith('p='):
                    self.p = int(line.split('=')[1].strip())
                elif line.startswith('g='):
                    self.g = int(line.split('=')[1].strip())
    
    def power(self, a, b, p):
        '''
        This function calculates (a^b) mod p using modular exponentiation.
        '''
        if b == 0:
            return 1
        elif b == 1:
            return a % p
        # else:
        #     return (pow(a, b) % p)

        # Optimized approach using exponentiation by squaring
        result = 1
        if a >= p:
            a = a % p
        while b > 0:
            # If b is odd, multiply a with the result
            if (b % 2) == 1:
                result = (result * a) % p
            # Now b must be even
            b //= 2 # Floor division (Integer division)
            a = (a * a) % p
        return result

    def compute_public_key(self, private_key):
        public_key = self.power(self.g, private_key, self.p)
        return public_key    

In [15]:
A = DiffieHellman()
B = DiffieHellman()

A.get_public_key()
B.get_public_key()

A.shared_key = A.power(A.g, A.private_key, A.p)
B.shared_key = B.power(B.g, B.private_key, B.p)

print(f'Shared key computed by A: {A.shared_key}')
print(f'Shared key computed by B: {B.shared_key}\n')

secret_key_A = A.power(B.shared_key, A.private_key, A.p)
secret_key_B = B.power(A.shared_key, B.private_key, B.p)

print(f'Secret key computed by A: {secret_key_A}')
print(f'Secret key computed by B: {secret_key_B}')

Shared key computed by A: 215842558092481
Shared key computed by B: 1910906368635147

Secret key computed by A: 51684233865755
Secret key computed by B: 51684233865755
